In [1]:
%load_ext autoreload
%autoreload 2

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
from tqdm import trange, tqdm
from itertools import product
import os

from testis_model.spatial_gillespie import SpatialAdvancedTestisModel
from testis_model.gillespie import sample_ic
import testis_model as tm

In [6]:
filepath = os.path.join('experiments', 'full_pipeline', 'final_data.csv')
all_data = pd.read_csv(filepath)

unique_fish = all_data['fish'].unique()
unique_fish = [f for f in unique_fish if 'ctrl' not in f]

fish_nums = sorted([int(f.replace('fish', '')) for f in unique_fish])
print(f'The unique fish found in the dataset are: {fish_nums}.')

def fish_metadata(all_data, fish):
    """
    This function takes in a loaded dataset and a fish identifier. It returns the times at which the fish
    was sampled as well as the initial distribution of clones in that fish. These values are used to
    initialize simulated fish with the same sample times and initial distribution.
    """
    fish_data = all_data.query('fish == @fish')

    # Get sample times.
    sample_ids = np.sort(fish_data['sample'].unique())
    sample_times = [tm.data_processing.sample_map_int[sample_id] for sample_id in sample_ids]

    # Find initial distribution.
    min_sample = fish_data['sample'].min()
    init_reads = np.sort(fish_data.query('sample == @min_sample') \
                         .groupby('barcode', as_index = False) \
                         .agg({'reads': 'sum'})['reads'])
    
    init_counts = tm.data_processing.normalize_tuple(init_reads, 1000)

    return sample_times, init_counts

The unique fish found in the dataset are: [1, 2, 4, 5, 8, 10, 12, 13, 14, 15, 16, 17, 18, 22, 26, 35, 37, 38, 40, 42, 44, 45, 50, 51].


In [9]:
data = []

fish = fish_nums[0]

# Init counts is mock fish, init freq is generated
sample_times, init_counts = fish_metadata(all_data, f'fish{fish}')

n_stem = 1000
n_clones = len(init_counts)
rng = np.random.default_rng(1)
init_freq = sample_ic(n_stem, n_clones, rng = rng)

sample_times_rep = np.repeat(sample_times, 1)
n_clones = len(init_counts)

i = 0

r_event = 36
n_stem = 1000
r_diff = 36
r_decay = 6.28
tau = 0.2
sperm_per_diff = 631
n_sims = 10
n_sperm = 470
p_diff = r_diff / r_event

seed = 1
rng = np.random.default_rng(seed)

hmm = SpatialAdvancedTestisModel(
    n_stem = n_stem, n_clones = n_clones, r_event = r_event, p_diff = p_diff,
    init_freq = init_counts, 
    r_decay = r_decay, sperm_per_diff = sperm_per_diff,
    tau = tau,
    c1 = 0.5, c2 = 0.5, n_tubules = 10, clone_choice = 'remaining'
)

record_times = np.arange(4, 16, 0.1)

sample_df, record_df, avg_stem_state, spatial_result = hmm.simulate_gillespie(
    sample_times_rep, rng=rng, n_sperm = n_sperm, init_time = 4,
    record_times=record_times,
)
# results.insert(0, 'scenario', scenario)
# results.insert(1, 'fish', fish)
# results.insert(2, 'duplicate', i)
# tm.data_processing.sort_results(results)
# data.append(results)

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8, 6))
gs = fig.add_gridspec(2, 3)

# Top row: full-width plot
ax_top = fig.add_subplot(gs[0, :])
tm.visualize.plot(sample_df, ax=ax_top)

# Bottom row: three tubule plots
ax1 = fig.add_subplot(gs[1, 0])
ax2 = fig.add_subplot(gs[1, 1])
ax3 = fig.add_subplot(gs[1, 2])

tm.visualize.plot_tubules(spatial_result, 5, ax=ax1)
tm.visualize.plot_tubules(spatial_result, 10, ax=ax2)
tm.visualize.plot_tubules(spatial_result, 16, ax=ax3)

plt.tight_layout()
plt.show()

In [144]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

times = spatial_result["record_times"]
states = spatial_result["record_states"]

fig = plt.figure(figsize=(10, 6))
gs = fig.add_gridspec(2, 1, height_ratios=[2, 1])

ax_top = fig.add_subplot(gs[0])
tm.visualize.plot(sample_df, ax=ax_top)

ax_bottom = fig.add_subplot(gs[1])
tm.visualize.plot_tubules(spatial_result, times[0], ax=ax_bottom)

patches = ax_bottom.patches
max_clone = max(np.max(tubule) for state in states for tubule in state)
palette = np.tile(
    tm.visualize.default_colors,
    -(-(max_clone + 1) // len(tm.visualize.default_colors))
)

def update(frame):
    clones = np.concatenate(states[frame])
    for patch, clone in zip(patches, clones):
        patch.set_facecolor(palette[clone])
    ax_bottom.set_title(f"Time {times[frame]:.1f}")
    return *patches, ax_bottom.title

animation = FuncAnimation(
    fig, update, frames=len(times), interval=50, blit=True
)

plt.tight_layout()
plt.close(fig)
HTML(animation.to_jshtml())

In [125]:
animation.save("tubules.gif", writer="pillow", fps=20)

In [88]:
sample_df.query('replicate == 1')

,time,replicate,obs_result,hid_result,full_result,n_sperm
0,5.0,1,"[4, 65, 159, 25, 141, 18, 39, 19]","[11, 131, 371, 43, 253, 73, 79, 39]","[60978, 608012, 1658635, 196912, 1271059, 3371...",470
3,6.0,1,"[4, 54, 191, 26, 98, 31, 44, 22]","[12, 141, 362, 45, 243, 76, 80, 41]","[57977, 614605, 1667759, 210958, 1167048, 3386...",470
6,9.0,1,"[5, 71, 168, 17, 109, 47, 32, 21]","[10, 155, 370, 38, 253, 63, 75, 36]","[37157, 672142, 1714525, 203711, 1220109, 3067...",470
9,10.0,1,"[2, 87, 159, 16, 123, 33, 36, 14]","[2, 175, 384, 39, 241, 56, 78, 25]","[14614, 752380, 1767055, 182383, 1152471, 2636...",470
12,11.0,1,"[4, 83, 182, 27, 99, 23, 40, 12]","[4, 186, 402, 39, 214, 49, 79, 27]","[30803, 820571, 1862250, 184807, 1003035, 2439...",470
15,12.0,1,"[3, 81, 191, 11, 117, 15, 32, 20]","[4, 184, 397, 33, 214, 52, 90, 26]","[23193, 845447, 1801826, 164224, 1022862, 2231...",470
18,14.0,1,"[0, 91, 175, 18, 104, 36, 40, 6]","[0, 172, 441, 39, 182, 62, 90, 14]","[44, 793201, 1994178, 170408, 871739, 273645, ...",470
21,16.0,1,"[0, 94, 204, 20, 79, 23, 47, 3]","[0, 181, 439, 41, 170, 62, 102, 5]","[0, 824322, 2021557, 190133, 815421, 274525, 4...",470
24,17.0,1,"[0, 81, 192, 16, 106, 24, 50, 1]","[0, 175, 470, 34, 159, 56, 103, 3]","[0, 794496, 2174122, 160369, 763940, 250392, 4...",470
27,18.0,1,"[0, 80, 226, 13, 64, 34, 51, 2]","[0, 170, 474, 44, 159, 61, 89, 3]","[0, 770473, 2186591, 190571, 692308, 269858, 4...",470
